In [1]:
import os
from dotenv import load_dotenv  # Carrega variáveis de ambiente do arquivo .env

import dspy  # Framework para construção de programas com LLMs
from pydantic import BaseModel, Field  # Modelos de dados estruturados

load_dotenv()  # Lê variáveis do arquivo .env


True

## Setup - Configuração do Modelo

Carregamos a chave da OpenAI a partir do arquivo `.env`, seguindo o mesmo padrão do notebook de referência.

Neste exemplo não definimos `temperature`, evitando parâmetros desnecessários na configuração do modelo.


In [2]:
lm = dspy.LM(
    "openai/gpt-5-mini",  # Mesmo modelo utilizado no notebook de referência
    api_key=os.getenv("OPENAI_API_KEY"),  # API key carregada do .env
)

# Configura o modelo padrão para todos os módulos DSPy
dspy.configure(lm=lm)


## Objetivo

Este notebook implementa um pequeno sistema de geração e avaliação de redações.

O fluxo será:

```text
                           TEMA
                            │
                            ▼
                    GerarRedacaoENEM
                            │
                            ▼
                         REDAÇÃO
                            │
            ┌───────────────┼─────────────────────────────┐
            │               │               │             │
            ▼               ▼               ▼             ▼
       Relevância       Gramática       Estrutura     Profundidade
            │               │               │             │
            └───────────────┼─────────────────────────────┘
                            │
                            ▼
                       
                   quatro avaliações
                            │
                            ▼
                  SintetizarAvaliacao
                            │
                            ▼
                    avaliação final
```

Os quatro avaliadores especializados são independentes. Por isso, serão executados com `dspy.Parallel`.

> **Observação:** os quatro critérios utilizados aqui formam uma avaliação experimental para demonstrar composição de módulos no DSPy. Eles não reproduzem as cinco competências oficiais do ENEM.


# 1. Gerador de Redação

Primeiro definimos uma `Signature` responsável por receber um tema e gerar uma redação dissertativo-argumentativa.

A `Signature` descreve **o que deve ser feito**. Em seguida, usamos `dspy.ChainOfThought` como estratégia de execução.


In [3]:
class GerarRedacaoENEM(dspy.Signature):
    """Gere uma redação dissertativo-argumentativa, em português formal, a partir do tema informado.

    A redação deve apresentar:
    - introdução com apresentação clara do tema e da tese;
    - desenvolvimento coerente dos argumentos;
    - progressão lógica entre os parágrafos;
    - conclusão compatível com o raciocínio desenvolvido;
    - proposta de intervenção quando adequada ao tema.

    Produza apenas uma redação completa e coerente, sem comentários sobre o processo de escrita.
    """

    # Entrada: tema que deverá orientar toda a redação
    tema: str = dspy.InputField(
        desc="Tema proposto para a redação."
    )

    # Saída: texto integral produzido pelo modelo
    redacao: str = dspy.OutputField(
        desc="Redação completa em formato dissertativo-argumentativo."
    )


## Criar o módulo gerador

Usaremos `dspy.ChainOfThought` para permitir que o modelo organize internamente a tarefa antes de produzir a redação.


In [4]:
# Cria o módulo responsável por gerar a redação
gerador_redacao = dspy.ChainOfThought(GerarRedacaoENEM)

## Definir o tema

Você pode alterar apenas esta variável para testar outros temas.


In [5]:
tema = (
    "A história da matemática"
)

print("TEMA:")
print(tema)


TEMA:
A história da matemática


## Gerar a redação

Passamos somente o tema. O módulo retorna uma `Prediction` contendo o campo `redacao` e, por ser um `ChainOfThought`, também pode conter o campo adicional de raciocínio do módulo.


In [6]:
resultado_geracao = gerador_redacao(
    tema=tema
)

resultado_geracao


Prediction(
    reasoning='Vou abordar a história da matemática como um processo cumulativo, multicultural e fundamental para o desenvolvimento científico, tecnológico e social. A redação terá introdução com apresentação do tema e tese (a matemática é produto humano coletivo e deve ser valorizada historicamente para melhorar ensino e cidadania); dois ou três parágrafos de desenvolvimento explorando origens antigas, contribuições de várias civilizações, transformação em disciplina abstrata e suas aplicações contemporâneas; proposta de intervenção concreta (reforma curricular, formação de professores, recursos pedagógicos e divulgação científica); e conclusão que retoma a tese e reforça a necessidade das ações propostas.',
    redacao='A história da matemática revela-se não apenas como um catálogo de teoremas e métodos, mas como um espelho das transformações culturais, econômicas e tecnológicas da humanidade. Desde as primeiras tábuas cuneiformes até os atuais algoritmos que movem a econ

## Visualizar a redação gerada


In [7]:
redacao = resultado_geracao.redacao

print("=" * 80)
print("REDAÇÃO GERADA")
print("=" * 80)
print(redacao)


REDAÇÃO GERADA
A história da matemática revela-se não apenas como um catálogo de teoremas e métodos, mas como um espelho das transformações culturais, econômicas e tecnológicas da humanidade. Desde as primeiras tábuas cuneiformes até os atuais algoritmos que movem a economia digital, a matemática tem sido construída de forma cumulativa e multicultural, tornando-se ferramenta essencial para o pensamento crítico e para a inovação. Defendo, portanto, que compreender e valorizar essa trajetória histórica é condição necessária para melhorar o ensino, combater preconceitos e fortalecer a capacidade de resolver problemas complexos na sociedade contemporânea.

As origens da matemática remontam às necessidades práticas das primeiras civilizações: a contagem e a medição para comércio e agricultura na Mesopotâmia e no Egito, os métodos algébricos pragmáticos na China e as soluções numéricas na Índia, que viria a introduzir o valor do zero e o sistema decimal. No mundo islâmico medieval, esses sab

# 2. Avaliador Composto

Agora construiremos o avaliador que motivou este exemplo.

A ideia é separar:

```text
Signature  → contrato de uma tarefa específica
BaseModel  → estrutura dos dados produzidos
Module     → composição e execução das várias tarefas
```

Cada avaliador especializado produzirá um objeto `Avaliacao`.


## Modelo de dados da avaliação

Todos os especialistas retornarão o mesmo formato.

Isso facilita a composição posterior porque relevância, gramática, estrutura e profundidade terão uma interface comum.


In [8]:
class Avaliacao(BaseModel):
    """Resultado estruturado produzido por um avaliador especializado."""

    nota: int = Field(
        ge=0,
        le=100,
        description="Nota do critério, entre 0 e 100."
    )

    justificativa: str = Field(
        description="Explicação objetiva e detalhada para a nota atribuída."
    )


## Avaliador 1 - Relevância

Este especialista recebe o **tema e a redação**, pois precisa verificar se o texto realmente responde ao tema proposto.


In [9]:
class AvaliarRelevancia(dspy.Signature):
    """Avalie exclusivamente a relevância da redação em relação ao tema proposto.

    Considere:
    - aderência ao tema;
    - manutenção do foco;
    - presença de uma tese relacionada ao problema;
    - ausência de fuga ou tangenciamento significativo do tema.

    Não avalie gramática, estrutura textual ou profundidade argumentativa,
    exceto quando esses aspectos afetarem diretamente a relevância.
    """

    tema: str = dspy.InputField(
        desc="Tema proposto para a redação."
    )

    redacao: str = dspy.InputField(
        desc="Texto completo da redação."
    )

    avaliacao: Avaliacao = dspy.OutputField(
        desc="Avaliação estruturada da relevância da redação."
    )


## Avaliador 2 - Gramática

Este especialista concentra-se na qualidade linguística da redação.


In [10]:
class AvaliarGramatica(dspy.Signature):
    """Avalie exclusivamente a qualidade gramatical e linguística da redação.

    Considere:
    - ortografia;
    - concordância verbal e nominal;
    - regência;
    - pontuação;
    - construção sintática;
    - adequação à norma-padrão da língua portuguesa.

    Não reduza a nota por discordar dos argumentos apresentados.
    """

    redacao: str = dspy.InputField(
        desc="Texto completo da redação."
    )

    avaliacao: Avaliacao = dspy.OutputField(
        desc="Avaliação estruturada da gramática da redação."
    )


## Avaliador 3 - Estrutura

Este especialista observa a organização global do texto e a relação entre suas partes.


In [11]:
class AvaliarEstrutura(dspy.Signature):
    """Avalie exclusivamente a estrutura e a organização da redação.

    Considere:
    - introdução;
    - desenvolvimento;
    - conclusão;
    - organização dos parágrafos;
    - progressão das ideias;
    - coesão entre as partes do texto.

    Não concentre a avaliação em erros gramaticais isolados.
    """

    redacao: str = dspy.InputField(
        desc="Texto completo da redação."
    )

    avaliacao: Avaliacao = dspy.OutputField(
        desc="Avaliação estruturada da organização da redação."
    )


## Avaliador 4 - Profundidade

Este especialista analisa a qualidade do desenvolvimento argumentativo.


In [12]:
class AvaliarProfundidade(dspy.Signature):
    """Avalie exclusivamente a profundidade argumentativa da redação.

    Considere:
    - qualidade e pertinência dos argumentos;
    - desenvolvimento das ideias;
    - capacidade de estabelecer relações de causa e consequência;
    - uso produtivo de repertório;
    - análise crítica em vez de afirmações superficiais.

    Não concentre a avaliação em correções gramaticais.
    """

    redacao: str = dspy.InputField(
        desc="Texto completo da redação."
    )

    avaliacao: Avaliacao = dspy.OutputField(
        desc="Avaliação estruturada da profundidade argumentativa."
    )


## Signature de Síntese

Os quatro especialistas produzem objetos do tipo `Avaliacao`.

A próxima `Signature` não executa os especialistas. Ela apenas **recebe os quatro resultados estruturados** e produz um parecer global.

Este é o ponto que diferencia:

```text
AvaliarGramatica      → tarefa
Avaliacao             → dado produzido pela tarefa
SintetizarAvaliacao   → tarefa que recebe os dados anteriores
```


In [13]:
class SintetizarAvaliacao(dspy.Signature):
    """Produza uma avaliação geral combinando os quatro pareceres especializados.

    Considere de forma equilibrada relevância, gramática, estrutura e profundidade.

    A nota final deve refletir o conjunto das quatro avaliações.
    O parecer final deve destacar os principais pontos fortes e os aspectos
    que mais limitam a qualidade da redação.
    """

    relevancia: Avaliacao = dspy.InputField(
        desc="Resultado do avaliador de relevância."
    )

    gramatica: Avaliacao = dspy.InputField(
        desc="Resultado do avaliador de gramática."
    )

    estrutura: Avaliacao = dspy.InputField(
        desc="Resultado do avaliador de estrutura."
    )

    profundidade: Avaliacao = dspy.InputField(
        desc="Resultado do avaliador de profundidade."
    )

    nota_final: int = dspy.OutputField(
        desc="Nota global da redação, entre 0 e 100.",
        ge=0,
        le=100,
    )

    parecer_final: str = dspy.OutputField(
        desc="Síntese geral da qualidade da redação."
    )


# 3. `AvaliadorGeral` como `dspy.Module`

Aqui ocorre a composição propriamente dita.

O `AvaliadorGeral` contém cinco submódulos:

```text
AvaliadorGeral
│
├── relevancia    → ChainOfThought(AvaliarRelevancia)
├── gramatica     → ChainOfThought(AvaliarGramatica)
├── estrutura     → ChainOfThought(AvaliarEstrutura)
├── profundidade  → ChainOfThought(AvaliarProfundidade)
│
└── sintetizar    → Predict(SintetizarAvaliacao)
```

Os quatro primeiros podem ser executados em paralelo porque nenhum depende do resultado dos demais.


In [14]:
class AvaliadorGeral(dspy.Module):
    """Composição dos avaliadores especializados e do sintetizador final."""

    def __init__(self):
        super().__init__()

        # ------------------------------------------------------------
        # Submódulos especializados
        # ------------------------------------------------------------

        self.relevancia = dspy.ChainOfThought(
            AvaliarRelevancia
        )

        self.gramatica = dspy.ChainOfThought(
            AvaliarGramatica
        )

        self.estrutura = dspy.ChainOfThought(
            AvaliarEstrutura
        )

        self.profundidade = dspy.ChainOfThought(
            AvaliarProfundidade
        )

        # ------------------------------------------------------------
        # Submódulo responsável pela síntese dos quatro resultados
        # ------------------------------------------------------------

        self.sintetizar = dspy.Predict(
            SintetizarAvaliacao
        )

        # ------------------------------------------------------------
        # Executor paralelo
        # ------------------------------------------------------------
        # Cada avaliador especializado é independente dos demais.
        # Portanto, podemos executar as quatro chamadas simultaneamente.
        self.parallel = dspy.Parallel(
            num_threads=4,
            max_errors=1,
            disable_progress_bar=False,
            timeout=120,
        )


    def forward(self, tema, redacao):

        # ------------------------------------------------------------
        # Define os pares:
        #
        #     (módulo, argumentos)
        #
        # que serão enviados ao dspy.Parallel.
        # ------------------------------------------------------------

        exec_pairs = [

            # Avaliador de relevância precisa receber também o tema.
            (
                self.relevancia,
                {
                    "tema": tema,
                    "redacao": redacao,
                }
            ),

            # Os demais avaliam características internas do texto.
            (
                self.gramatica,
                {
                    "redacao": redacao,
                }
            ),

            (
                self.estrutura,
                {
                    "redacao": redacao,
                }
            ),

            (
                self.profundidade,
                {
                    "redacao": redacao,
                }
            ),
        ]

        # ------------------------------------------------------------
        # Executa os quatro especialistas em paralelo.
        # ------------------------------------------------------------

        resultados = self.parallel(
            exec_pairs
        )

        # A ordem dos resultados acompanha a ordem dos exec_pairs.
        resultado_relevancia = resultados[0]
        resultado_gramatica = resultados[1]
        resultado_estrutura = resultados[2]
        resultado_profundidade = resultados[3]

        # ------------------------------------------------------------
        # Extrai apenas o objeto estruturado Avaliacao de cada resultado.
        # ------------------------------------------------------------

        relevancia = resultado_relevancia.avaliacao
        gramatica = resultado_gramatica.avaliacao
        estrutura = resultado_estrutura.avaliacao
        profundidade = resultado_profundidade.avaliacao

        # ------------------------------------------------------------
        # A quinta Signature recebe os quatro resultados.
        # ------------------------------------------------------------

        sintese = self.sintetizar(
            relevancia=relevancia,
            gramatica=gramatica,
            estrutura=estrutura,
            profundidade=profundidade,
        )

        # ------------------------------------------------------------
        # Um dspy.Module deve retornar preferencialmente uma Prediction.
        #
        # Assim, o chamador recebe tanto as quatro avaliações
        # especializadas quanto a síntese final em um único objeto.
        # ------------------------------------------------------------

        return dspy.Prediction(
            relevancia=relevancia,
            gramatica=gramatica,
            estrutura=estrutura,
            profundidade=profundidade,
            nota_final=sintese.nota_final,
            parecer_final=sintese.parecer_final,
        )


## O que o `forward()` está fazendo?

Conceitualmente:

```text
                           redação
                              │
              ┌───────────────┼────────────────────────────┐
              │               │               │            │
              ▼               ▼               ▼            ▼
        relevancia        gramatica       estrutura    profundidade
              │               │               │            │
              └───────────────┼────────────────────────────┘
                              │
                              ▼

                 dspy.Parallel executa os
                quatro módulos simultaneamente

                              │
               ┌──────────────┼────────────────────────────┐
               ▼              ▼              ▼             ▼
           Avaliacao      Avaliacao      Avaliacao     Avaliacao
               │              │              │             │
               └──────────────┼────────────────────────────┘
                              │
                              ▼
                   SintetizarAvaliacao
                              │
                              ▼
                    nota + parecer final
```

A `Signature` descreve cada tarefa. O `Module` descreve o **programa** que conecta essas tarefas.


## Criar o avaliador geral


In [15]:
# Instancia todo o programa de avaliação
avaliador = AvaliadorGeral()


## Avaliar a redação gerada

Agora usamos diretamente a redação produzida na primeira parte do notebook.


In [16]:
resultado_avaliacao = avaliador(
    tema=tema,
    redacao=redacao,
)

resultado_avaliacao


Processed 4 / 4 examples: 100%|█| 4/4 [


Prediction(
    relevancia=Avaliacao(nota=98, justificativa='O texto aborda de forma consistente e direta a história da matemática, apresenta tese explícita e a desenvolve mantendo o foco no tema; os exemplos históricos, a transição para matemática teórica e as implicações para ensino e sociedade estão articulados com a tese, sem tangenciamento significativo. As propostas finais vinculam-se ao argumento central e reforçam a relevância do conteúdo histórico. Pequena dedução por ampliação do escopo para políticas educacionais (ainda que pertinente).'),
    gramatica=Avaliacao(nota=98, justificativa="A redação está praticamente isenta de erros gramaticais. Ortografia, concordância verbal e nominal, regência, pontuação e construção sintática obedecem à norma-padrão. As poucas observações são meramente estilísticas (por exemplo, variações possíveis como 'entre os alunos', 'veio a introduzir' ou a inclusão opcional do artigo em 'uma ferramenta'), que não comprometem a correção. Dada a qualid

## Ver as avaliações especializadas

Cada atributo abaixo é um objeto `Avaliacao`, com:

```text
nota
justificativa
```


In [17]:
def mostrar_avaliacao(nome, avaliacao):
    print("\n" + "=" * 80)
    print(nome.upper())
    print("=" * 80)
    print(f"Nota: {avaliacao.nota}/100")
    print("\nJustificativa:")
    print(avaliacao.justificativa)


mostrar_avaliacao(
    "Relevância",
    resultado_avaliacao.relevancia
)

mostrar_avaliacao(
    "Gramática",
    resultado_avaliacao.gramatica
)

mostrar_avaliacao(
    "Estrutura",
    resultado_avaliacao.estrutura
)

mostrar_avaliacao(
    "Profundidade",
    resultado_avaliacao.profundidade
)



RELEVÂNCIA
Nota: 98/100

Justificativa:
O texto aborda de forma consistente e direta a história da matemática, apresenta tese explícita e a desenvolve mantendo o foco no tema; os exemplos históricos, a transição para matemática teórica e as implicações para ensino e sociedade estão articulados com a tese, sem tangenciamento significativo. As propostas finais vinculam-se ao argumento central e reforçam a relevância do conteúdo histórico. Pequena dedução por ampliação do escopo para políticas educacionais (ainda que pertinente).

GRAMÁTICA
Nota: 98/100

Justificativa:
A redação está praticamente isenta de erros gramaticais. Ortografia, concordância verbal e nominal, regência, pontuação e construção sintática obedecem à norma-padrão. As poucas observações são meramente estilísticas (por exemplo, variações possíveis como 'entre os alunos', 'veio a introduzir' ou a inclusão opcional do artigo em 'uma ferramenta'), que não comprometem a correção. Dada a qualidade lingüística e a quase inexi

## Ver a avaliação final

A síntese recebe as quatro avaliações especializadas e produz uma visão global.


In [18]:
print("\n" + "=" * 80)
print("AVALIAÇÃO FINAL")
print("=" * 80)

print(
    f"Nota final: "
    f"{resultado_avaliacao.nota_final}/100"
)

print("\nParecer final:")
print(
    resultado_avaliacao.parecer_final
)



AVALIAÇÃO FINAL
Nota final: 95/100

Parecer final:
Redação de alta qualidade: trata o tema com precisão e foco, expõe tese clara e a desenvolve coerentemente com repertório histórico bem escolhido. A linguagem segue a norma-padrão quase sem falhas, demonstrando controle gramatical e estilístico. A organização textual é eficaz — introdução, desenvolvimento temático e conclusão funcionam bem, com transições adequadas e propostas finais articuladas ao argumento central. A profundidade é sólida: há conexão plausível entre evolução histórica, implicações para o ensino e propostas institucionais concretas.

Principais pontos fortes
- Clareza e foco temático; exemplos históricos pertinentes que sustentam a tese.
- Excelente domínio gramatical e estilístico.
- Estrutura lógica e coesa com propostas bem integradas ao desenvolvimento.

Aspectos que limitam a nota
- Abordagem levemente ampla ao incorporar políticas educacionais, o que exige evidências empíricas adicionais.
- Falta de tratamento 

## Resumo das notas

Podemos também visualizar as quatro notas de forma compacta.


In [19]:
notas = {
    "Relevância": resultado_avaliacao.relevancia.nota,
    "Gramática": resultado_avaliacao.gramatica.nota,
    "Estrutura": resultado_avaliacao.estrutura.nota,
    "Profundidade": resultado_avaliacao.profundidade.nota,
}

for criterio, nota in notas.items():
    print(f"{criterio:<15}: {nota:>3}/100")

print("-" * 30)
print(
    f"{'Nota final':<15}: "
    f"{resultado_avaliacao.nota_final:>3}/100"
)


Relevância     :  98/100
Gramática      :  98/100
Estrutura      :  95/100
Profundidade   :  90/100
------------------------------
Nota final     :  95/100


## Inspecionando o Histórico

Como o programa faz várias chamadas ao modelo, podemos inspecionar as chamadas recentes.

Neste fluxo teremos, em geral:

1. geração da redação;
2. relevância;
3. gramática;
4. estrutura;
5. profundidade;
6. síntese final.

A ordem em que as quatro avaliações intermediárias aparecem no histórico pode variar por causa da execução paralela.


In [21]:
# Mostra as últimas chamadas realizadas ao modelo
dspy.inspect_history(n=20)






[2026-09-07T14:17:56.960702]

System message:

Your input fields are:
1. `tema` (str): Tema proposto para a redação.
Your output fields are:
1. `reasoning` (str): 
2. `redacao` (str): Redação completa em formato dissertativo-argumentativo.
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## tema ## ]]
{tema}

[[ ## reasoning ## ]]
{reasoning}

[[ ## redacao ## ]]
{redacao}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Gere uma redação dissertativo-argumentativa, em português formal, a partir do tema informado.
        
        A redação deve apresentar:
        - introdução com apresentação clara do tema e da tese;
        - desenvolvimento coerente dos argumentos;
        - progressão lógica entre os parágrafos;
        - conclusão compatível com o raciocínio desenvolvido;
        - proposta de intervenção quando adequada ao tema.
        
        Produza apenas uma redação completa e coerente,